# GReaT Synthetic v1 Template

A Colab-ready template for generating a synthetic v1 dataset with `be_great` on any tabular CSV dataset. Change the dataset configuration cell when reusing this notebook for a new dataset.

## Configuration

Edit the following cell for a new dataset. The most common changes are `DATASET_NAME` and `DATASET_CSV`.

In [ ]:
# === Edit these lines for a new dataset ===
DATASET_NAME = "california"
DATASET_DISPLAY_NAME = "California Housing"
DATASET_KIND = "california_housing"
DATASET_CSV = "/content/california.csv"
LLM_NAME = "distilgpt2"
EPOCHS = 5
BATCH_SIZE = 16
MAX_ROWS = 5000
N_SAMPLES = 5000
FLOAT_PRECISION = 5
SEED = 42
TEST_SIZE = 0.2
OUTPUT_ROOT = Path("/content") / DATASET_NAME / "outputs" / "synthetic_v1"
LOG_DIR = OUTPUT_ROOT / "log"

## 1. Install Colab Dependencies

Install `be_great` and the supporting packages used by this template.

In [ ]:
!pip -q install git+https://github.com/tabularis-ai/be_great.git pandas scikit-learn

## 2. Import Libraries

Import the libraries needed for dataset loading, training, generation, and file handling.

In [ ]:
import json
import random
from pathlib import Path

import pandas as pd
import torch
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from be_great import GReaT

## 3. Load Dataset

Load any tabular CSV dataset, keep a configurable training subset, and prepare the output paths.

In [ ]:
if DATASET_KIND == "california_housing":
    california = fetch_california_housing()
    df = pd.DataFrame(california.data, columns=california.feature_names)
    df["MedHouseVal"] = california.target
elif DATASET_KIND == "csv":
    df = pd.read_csv(DATASET_CSV)
else:
    raise ValueError(f"Unsupported DATASET_KIND: {DATASET_KIND}")

original_rows = len(df)
train_df, test_df = train_test_split(df, test_size=TEST_SIZE, random_state=SEED, shuffle=True)
train_rows = len(train_df)
sample_count = N_SAMPLES

output_dir = OUTPUT_ROOT
log_dir = LOG_DIR
output_dir.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)
experiment_dir = log_dir

print(f"Loaded dataset: {DATASET_DISPLAY_NAME}")
print(f"Original rows: {original_rows}")
print(f"Training rows: {train_rows}")
print(f"Test rows: {len(test_df)}")
print(f"Synthetic rows to generate: {sample_count}")
print(f"Outputs will be saved in: {output_dir}")

## 4. Initialize GReaT

Create the `be_great` model using `distilgpt2`.

In [ ]:
great = GReaT(
    LLM_NAME,
    experiment_dir=str(experiment_dir),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    float_precision=FLOAT_PRECISION,
)
print(great)

## 5. Train GReaT

Fit the model on the selected training subset.

In [ ]:
great.fit(train_df)

## 6. Generate Synthetic v1

Sample synthetic rows using `be_great` and ensure the output is at least as large as the original dataset.

In [ ]:
try:
    synthetic_df = great.sample(
        n_samples=sample_count,
        guided_sampling=True,
        device="auto",
    )
except Exception as exc:
    print(f"guided_sampling failed: {exc}")
    print("Falling back to legacy sampling...")
    synthetic_df = great.sample(
        n_samples=sample_count,
        guided_sampling=False,
        device="auto",
    )

print(synthetic_df.head())
print(f"Generated rows: {len(synthetic_df)}")

## 7. Post-process and Save Results

Clean the generated dataset, then save the CSV and metadata into the dataset-specific output folder.

In [ ]:
synthetic_df = synthetic_df.drop_duplicates().reset_index(drop=True)

output_csv = output_dir / "cal_synthetic_v1.csv"
output_metadata = log_dir / "cal_synthetic_v1_metadata.json"

synthetic_df.to_csv(output_csv, index=False)
metadata = {
    "dataset": DATASET_DISPLAY_NAME,
    "model": LLM_NAME,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "training_rows": int(len(train_df)),
    "synthetic_rows": int(len(synthetic_df)),
    "seed": SEED,
}
output_metadata.write_text(json.dumps(metadata, indent=2))

print(f"Saved CSV to: {output_csv}")
print(f"Saved metadata to: {output_metadata}")
print(metadata)